# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the FAIR$^2$ rangeland knowledge adoption dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all entities by their `@id`s for consistency and reproducibility.

### Dataset Source
The dataset Croissant schema is available at:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

*Citation: Kamadi, V, Chimoita, EL, Wahome, RG and Odhong, C 2026 Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya (Frontiers)*

In [ ]:
# Make sure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We'll load the Croissant schema, retrieve the dataset metadata, and print a short summary.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Accessing metadata (as a dataclass, not as a dictionary!)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

# Print publication date, coverage, available record sets, and more for overview
print(f"Date published: {getattr(metadata, 'datePublished', '-')}")
print(f"Spatial coverage: {getattr(metadata, 'spatialCoverage', '-')}")
print(f"Temporal coverage: {getattr(metadata, 'temporalCoverage', '-')}")
print(f"Keywords: {getattr(metadata, 'keywords', '-')}")

## 2. Data Overview

List all available record sets (`cr:RecordSet`) in the dataset with their `@id` and their respective fields/columns. _All entity references use their `@id` values._

In [ ]:
# List available record sets in the dataset by their @id and included field @ids
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in this Croissant metadata. Please check the schema for available record sets.")
else:
    print(f"Found {len(record_sets)} Record Sets:")
    for rset in record_sets:
        print(f"- RecordSet @id: {rset['@id']}")
        if 'field' in rset:
            # rset['field'] may be a list or dict
            if isinstance(rset['field'], list):
                field_ids = [field['@id'] if isinstance(field, dict) and '@id' in field else str(field) for field in rset['field']]
            else:
                field_ids = [rset['field']['@id']] if isinstance(rset['field'], dict) and '@id' in rset['field'] else [str(rset['field'])]
            print("    Fields/Columns @id:", field_ids)
        else:
            print("    (No fields listed)")

## 3. Data Extraction

Use the record set `@id`s found above to extract tables of data.

**Note**: If no record sets are listed, try running `dataset.records()` with no arguments to see what's available, or inspect distributions/encoding info via metadata.

In [ ]:
# Step 1: Get list of all available record_set @ids
available_record_set_ids = [rset['@id'] for rset in dataset.record_sets] if dataset.record_sets else []

if not available_record_set_ids:
    print("No record_sets with explicit @id structure found in metadata. Attempting to load available records generically...")
    # fallback: try dataset.records() (no record_set arg)
    sample_records = list(dataset.records())
    print(f"Sample records (first 2): {sample_records[:2]}")
    # Attempt to make a DataFrame
    if sample_records:
        df = pd.DataFrame(sample_records)
        print("Column names:", df.columns.tolist())
        display(df.head())
    else:
        print("No records could be loaded from this dataset.")
else:
    print("Record set @ids detected:", available_record_set_ids)
    dataframes = {}
    for record_set_id in available_record_set_ids:
        print(f"\nLoading records for record_set @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame with shape: {df.shape}")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head(3))
        else:
            print("No records found for this record set.")
    # For demonstration: pick the first record_set loaded (if any) for EDA below
    if dataframes:
        chosen_record_set_id = list(dataframes.keys())[0]
        print(f"\n`chosen_record_set_id` for EDA: {chosen_record_set_id}")
    else:
        chosen_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

Let's analyze a numeric field (by its `@id`), show basic statistics, filter for a threshold, normalize, and group by a categorical field. _Ensure field references use their `@id`s._

In [ ]:
# This block demonstrates numeric filtering and grouping EDA using @ids for fields

import numpy as np

# Choose a record set to analyze (from previous block)
if 'dataframes' in globals() and dataframes:
    record_set_id = chosen_record_set_id
    df = dataframes[record_set_id]
else:
    print("No dataframes found. Please ensure you have successfully loaded data in step 3.")
    df = None

if df is not None and not df.empty:
    # Show numeric fields candidates (float, int)
    numeric_candidates = []
    for col in df.columns:
        if np.issubdtype(df[col].dropna().dtype, np.number):
            numeric_candidates.append(col)
    print('Numeric candidates by @id:', numeric_candidates)
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Pick first numeric field for demonstration (by @id)
        print(f"Using numeric field: {numeric_field_id}")

        # Choose a threshold suitably small for demo if possible (use 10 or data minimum+epsilon)
        try:
            min_val = df[numeric_field_id].min()
            threshold = min(10, min_val+1) if min_val < 10 else min_val  # heuristics
        except Exception:
            threshold = 10

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (shape: {filtered_df.shape}):")
        display(filtered_df.head())

        # Normalize the numeric field
        mean_v = filtered_df[numeric_field_id].mean()
        std_v = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_v) / std_v if std_v > 0 else 0
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Show grouping by a categorical field (pick first non-numeric column)
        group_candidates = [col for col in df.columns if not np.issubdtype(df[col].dropna().dtype, np.number)]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Grouped mean by", group_field_id)
            display(grouped_df.head())
        else:
            print("No categorical/group field candidates found.")
    else:
        print("No numeric columns for EDA demo.")
else:
    print("No data available for EDA.")

## 5. Visualization

Let's plot the filtered and normalized numeric variable, and visualize a grouped summary. All field references are by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and 'filtered_df' in locals():
    # Histogram of numeric values
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped summary is available, use it
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df.head(10))
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No filtered/grouped data available for visualization.")

## 6. Conclusion

* This notebook demonstrated programmatic exploration and processing of a Croissant-schema dataset, strictly referencing all data structures and fields by their `@id` as per best practice for ML dataset reproducibility.
* We inspected the dataset's record sets and fields, extracted and summarized numeric variables, filtered and normalized data, and visualized simple relationships.
* For further analysis, consult the full [mlcroissant documentation](https://mlcroissant.readthedocs.io/) and the [FAIR2 dataset article](https://sen.science/doi/10.71728/senscience.y7m0-f273/).

Remember: Always reference dataset fields by their canonical `@id` for clarity and version control.